In [ ]:
"""
Final cleanup pass: catch pure FORMATTING variants (hyphen presence,
whitespace, casing) that slipped through the main entity resolution
pipeline (step1/step2/step3). Confirmed real on this corpus: "water
holding capacity" and "water-holding capacity" both survived as
separate nodes despite going through the full review.

This is deliberately NOT another embedding+LLM pass. Hyphenation/
spacing/casing differences have been confirmed safe to merge every
single time they came up in this entire review (never once found a
case where a hyphen alone carried real meaning, unlike genotype
codes, sample letters, or state words like "gel"). So this catches
them directly and deterministically: normalize every entity the same
way, group anything that normalizes identically, merge automatically.

No LLM, no API cost, no manual review needed, this is a narrow,
low-risk safety net for exactly one failure mode, not a replacement
for the main pipeline.

Designed for Jupyter/Colab execution. No __main__ guard.
"""

import re
import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = r"../PART-5-Entity-Type-Resolution/type_resolved_triples_v2.xlsx"
SOURCE_COL = "resolved_source"
TARGET_COL = "resolved_target"
DOI_COL = "doi"

OUTPUT_TRIPLES_XLSX = "final_formatting_cleaned_triples.xlsx"
OUTPUT_REVIEW_XLSX = "formatting_variant_review.xlsx"


# ---------------------------------------------------------------
# NORMALIZE: hyphen/whitespace/casing only, nothing semantic
# ---------------------------------------------------------------
def normalize(entity):
    text = str(entity).lower().strip()
    text = text.replace("-", " ").replace("\u2013", " ").replace("\u2014", " ")  # hyphen, en dash, em dash
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples")

all_entities = pd.concat([df[SOURCE_COL], df[TARGET_COL]]).dropna().astype(str).str.strip().unique()
print(f"Unique entities: {len(all_entities)}")

# ---------------------------------------------------------------
# GROUP BY NORMALIZED FORM
# ---------------------------------------------------------------
norm_to_raw = {}
for e in all_entities:
    norm_to_raw.setdefault(normalize(e), []).append(e)

variant_groups = {k: v for k, v in norm_to_raw.items() if len(v) > 1}
print(f"\nFound {len(variant_groups)} groups of pure formatting variants:")
for norm, raws in list(variant_groups.items())[:15]:
    print(f"  {raws}")
if len(variant_groups) > 15:
    print(f"  ...and {len(variant_groups) - 15} more")

# ---------------------------------------------------------------
# BUILD LOOKUP: canonical = most frequent raw form (by doc frequency),
# ties broken by longer/more complete string
# ---------------------------------------------------------------
entity_doc_freq = pd.concat([
    df[[SOURCE_COL, DOI_COL]].rename(columns={SOURCE_COL: "entity"}),
    df[[TARGET_COL, DOI_COL]].rename(columns={TARGET_COL: "entity"}),
]).groupby("entity")[DOI_COL].nunique().to_dict()

lookup = {}
review_rows = []
for norm, raws in variant_groups.items():
    canonical = max(raws, key=lambda r: (entity_doc_freq.get(r, 0), len(r)))
    for r in raws:
        if r != canonical:
            lookup[r] = canonical
    review_rows.append({
        "variants": "; ".join(raws),
        "canonical_chosen": canonical,
        "doc_frequencies": "; ".join(f"{r}={entity_doc_freq.get(r, 0)}" for r in raws),
    })

# ---------------------------------------------------------------
# APPLY
# ---------------------------------------------------------------
df[SOURCE_COL] = df[SOURCE_COL].apply(lambda e: lookup.get(str(e).strip(), e))
df[TARGET_COL] = df[TARGET_COL].apply(lambda e: lookup.get(str(e).strip(), e))

n_nodes_before = len(all_entities)
n_nodes_after = pd.concat([df[SOURCE_COL], df[TARGET_COL]]).dropna().astype(str).str.strip().nunique()
print(f"\nNodes before: {n_nodes_before}, after: {n_nodes_after}, collapsed: {n_nodes_before - n_nodes_after}")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
df.to_excel(OUTPUT_TRIPLES_XLSX, index=False)
pd.DataFrame(review_rows).to_excel(OUTPUT_REVIEW_XLSX, index=False)
print(f"\nSaved cleaned triples to {OUTPUT_TRIPLES_XLSX}")
print(f"Saved review of what was merged to {OUTPUT_REVIEW_XLSX}")

Loaded 10324 triples
Unique entities: 2759

Found 9 groups of pure formatting variants:
  ['water holding capacity', 'water-holding capacity']
  ['freeze drying', 'freeze-drying']
  ['dry heating', 'dry-heating']
  ['random coil content', 'random-coil content']
  ['hot pressing', 'hot-pressing']
  ['cold pressing', 'cold-pressing']
  ['beta-turn content', 'beta turn content']
  ['Tan δ', 'tan δ']
  ['protein-water interactions', 'protein water interactions']

Nodes before: 2759, after: 2750, collapsed: 9

Saved cleaned triples to final_formatting_cleaned_triples.xlsx
Saved review of what was merged to formatting_variant_review.xlsx
